# CFPB Complaint Dataset — Exploratory Data Analysis

This notebook explores the processed sample dataset. Run `scripts/build_dataset.py` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cfpb_assistant.data.sampler import load_split
from cfpb_assistant.utils.config import resolve_path, load_config

sns.set_theme(style='whitegrid')
cfg = load_config('preprocessing')

In [ ]:
# Load the sample
sample_path = resolve_path(cfg['data']['samples_dir']) / 'complaints_sample.parquet'
df = pd.read_parquet(sample_path)
print(f'Sample shape: {df.shape}')
df.head(3)

In [ ]:
# Column overview
df.info()

In [ ]:
# Product label distribution
fig, ax = plt.subplots(figsize=(10, 5))
df['product_clean'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Product Label Distribution (sample)')
ax.set_xlabel('Product')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Narrative length distribution
df['narrative_len'] = df['narrative_clean'].str.len()
fig, ax = plt.subplots(figsize=(10, 4))
df['narrative_len'].clip(upper=2000).hist(bins=60, ax=ax)
ax.set_title('Narrative Length Distribution (clipped at 2000 chars)')
ax.set_xlabel('Character count')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()
df['narrative_len'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
# Missing values overview
missing = df.isnull().sum().sort_values(ascending=False)
print('Missing values per column:')
print(missing[missing > 0])

In [ ]:
# Year-over-year complaint volume (from sample)
if 'date_received' in df.columns:
    df['year'] = pd.to_datetime(df['date_received'], errors='coerce').dt.year
    fig, ax = plt.subplots(figsize=(10, 4))
    df['year'].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title('Complaint Volume by Year (sample)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Sample complaint narratives
for _, row in df[df['narrative_clean'].str.len() > 200].sample(3, random_state=42).iterrows():
    print(f"--- Product: {row['product_clean']} | Issue: {row.get('issue_clean', 'N/A')} ---")
    print(row['narrative_clean'][:400])
    print()